# Zürich Tram Flow — Analyse-Report

**93.9 Mio. Datenpunkte · 3 Jahre (2023–2025) · 16 Tramlinien · Zürich**

Dieser Report fasst die wichtigsten Erkenntnisse der Analyse-Phase zusammen.
Er richtet sich an alle die verstehen wollen, wo, wann und warum Trams in Zürich verspätet sind —
und was das für den Betrieb bedeutet.

## Datenbasis

| | |
|:---|:---|
| **Quelle** | VBZ Zürich — IST-Daten opentransportdata.swiss |
| **Zeitraum** | Januar 2023 – Oktober 2025 |
| **Umfang** | 93.9 Mio. Zeilen · 16 Tramlinien · 24 Features |
| **Analysebasis (lf_clean)** | canceled==False · stop_sequence>1 · ohne Linie E · ohne Nov/Dez 2025 |
| **OTP-Schwellwert** | ±120s (VBZ-Standard) |

Alle Zahlen in diesem Report basieren auf der bereinigten Datenbasis (lf_clean) — ausser wo explizit anders angegeben.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
from zh_tram_flow.config import PATHS
import zh_tram_flow.analytics as an

%load_ext autoreload
%autoreload 2

TRAIN, TEST, _ = setup_analysis("04_insights")

lf_all = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])

lf = (
    lf_all
    .filter(pl.col("canceled") == False)
    .filter(pl.col("stop_sequence") > 1)
    .filter(pl.col("line_name") != "E")
    .filter(
        ~((pl.col("operating_date").dt.year() == 2025) &
          (pl.col("operating_date").dt.month() >= 11))
    )
)

## 1. Das System — strukturelle Pufferschwäche

Das Zürcher Tramnetz fährt mit einer systemischen Schwäche: **71.5% aller Halte akkumulieren Verspätung** — kein Puffer ist eingebaut. Die Tram wächst sich also von Halt zu Halt in die Verspätung rein, statt sie abzubauen. Die OTP (On-Time Performance) liegt bei 87% — stabil, aber strukturell fragil.

In [ ]:
an.plot_dwell_time(lf)
show_df(an.table_dwell_time_by_line(lf))

## 2. Wo entstehen Verspätungen?

Die Hotspots sind **periphere Aussenkorridore** — nicht die zentralen Knotenpunkte. Friedhof Enzenbühl (93.8s), Balgrist (85.2s) und Leutschenbach (82.7s) führen die Liste an. Central und Paradeplatz, wo 14–15 Linien kreuzen, liegen beide unter dem Netzschnitt. Stadtkreis 11 (68.3s, OTP 83%) ist der schlechteste Kreis — Kreis 5 der beste (49.9s, OTP 89%).

In [ ]:
an.plot_stop_delay_map(lf)
an.plot_district_analysis(lf)

## 3. Wann entstehen Verspätungen?

**Kein Morgenrush.** Die Stunde 7h liegt mit 48.9s unter dem Netzschnitt. Der eigentliche Peak ist um 21h (67.9s) — Abreisewellen nach Konzerten und Spielen. Donnerstag ist der schlechteste Wochentag (60.4s), Sonntag der beste (48.4s). November ist konsistent der schlechteste Monat (2024: 72.6s) — Winter überraschend die beste Jahreszeit (51.7s, OTP 88.9%).

In [ ]:
an.plot_hour_of_day(lf)
an.plot_day_of_week(lf)
an.plot_month_seasonality(lf)

## 4. Wettereinflüsse — Schnee ist der stärkste Faktor

**Schnee** ist der stärkste Einzeleinflussfaktor im gesamten Datensatz: +54s Mehrdelay, OTP −10.9 Prozentpunkte. Und die geografische Trennung ist klar: Schnee trifft Höhenlagen (Kreise 10/4/12), Regen trifft Flusstäler (Kreis 5). Linien reagieren komplett unterschiedlich — L17 leidet stark unter Regen (+41.2s), L9 stark unter Schnee (+75.9s). Kälte (0–5°C) ist überraschend die **beste** Wetterbedingung (53.8s) — die Frost-Hypothese ist falsch.

In [ ]:
an.plot_weather_overview(lf)
an.plot_weather_stop_map(lf, flag="has_snow")

## 5. Events & Feiertage

**Feiertage sind die besten Tage** — 46.3s, −9.9s gegenüber Normal. Der Rückgang des Berufsverkehrs überwiegt jeden Eventeffekt. Grosse Events hingegen kosten +10.5s — aber fast ausschliesslich abends zwischen 18 und 22 Uhr. Tagsüber ist kein Unterschied messbar. Überraschung: **Fachmessen** (66.0s, OTP 84%) sind die schlechteste Ereigniskategorie — nicht Konzerte oder Fussball.

In [ ]:
an.plot_events_overview(lf)
an.plot_event_type_hourly_profile(lf)

## 6. Das Netz — Ausbau ohne Wirkung an den richtigen Orten

Der Fahrplanwechsel Dezember 2023 war der **grösste in der VBZ-Geschichte**: L9, L11 und L13 fundamental umgebaut. Im Delay-Signal ist das netzweit unsichtbar — +0.5s. Noch entscheidender: Die echten Streckenerweiterungen gingen nach Sihlcity (K3) und Rehalp (K8) — beide gut performende Gebiete. Kreise 11 und 12, die eigentlichen Problemzonen, erhielten nichts.

In [ ]:
an.plot_service_quality_district_map(lf_all)

## Fazit & Empfehlungen

Das Zürcher Tramnetz hat strukturelle Verspätungsmuster die über drei Jahre stabil sind. Die Analyse zeigt drei klare Handlungsfelder:

| Priorität | Empfehlung | Basis |
|:---|:---|:---|
| **1 — Hoch** | Fahrplanpuffer gezielt in Aussenkorridoren einbauen — K11/K12 brauchen mehr Haltezeit-Reserven, nicht mehr Linien | F-SPAT-01, F-TARGET-03 |
| **2 — Mittel** | Abend-Kapazität prüfen (18–22h) — Events + Feierabend treffen gleichzeitig. Sonderkonzept für Donnerstag-Abend prüfen | F-TEMP-01/02, F-EVNT-03 |
| **3 — Mittel** | Netzinvestitionen auf Hotspot-Kreise ausrichten — zukünftige Erweiterungen zuerst in K11/K12 evaluieren, nicht in bereits gut performenden Gebieten | F-NET-09, F-SPAT-03 |

**Was kommt:** Ein Vorhersagemodell (LightGBM) soll auf Basis dieser Features quantifizieren, welche Faktoren wie viel zur Verspätung beitragen — und welche Halte zu welcher Zeit am stärksten gefährdet sind.

## Export

```bash
jupyter nbconvert --to html \
  --no-input \
  --output-dir ../reports \
  --output index \
  04_insights.ipynb
```

`--no-input` blendet alle Code-Zellen aus — der Report zeigt nur Text und Charts.